### Neuronalen Netze und simultane Hyperparameteroptimierung 

Dieses Notebook dient dem Anknüpfen der Erkenntnisse aus 05_flo_NN, 05_Neural_Network_Felix, 05_Neural_Network_Koja und einer Simultan-Optimierung der Hyperparameter sowie weiterem Featureengeneering.
Als nächster logischer Schritt bietet sich aufbauend auf den Erkenntnissen von Felix H. die Analyse durch Bayesianische Optimierung.


### Bayesianische Optimierung 
Die Idee: 
Beim Training eines neuronalen Netzes sind zuvor Auswirkungen der Änderungen der Kernrate und Neuronenanzahl auf den Loss unbekannt -> Blackbox. Die Optimierung über den [keras_tuner] nutzt ein statistisches Modell (Gauß-Prozess Q2) un schätzt für jede Hyperparameterkombination die erwartete Treffergenauigkeit und die Unsicherheit des Modells. Die Akquisitionsfunktion entscheidet über den als nächstes getesteten Punkt und wägt dabei zwischen Nähe zu vielverprechenden Werten und der Erkundung neuer Bereiche mit hoher Unsicherheit ab.  
Der Keras BayesianOptimization-Tuner testet zunächst zufällige Hyperparameterkombinationen. Dann berechnet er vielverprechende Kombinationen aus den bisherigen Ergebnissen. Mit diesen Prametern wird ein Netz trainiert und das Ergebnis den vorigen Ergebnissen hinzugefügt. Anschlißend beginnt die Berechnung der vielverprechenden Kombinationen erneut bis max-trials erreicht ist.   
Der Geschwindigkeitszuwachs erfolgt dabei durch die Möglichkeit des Abbruchs nach weniger als der maximalen Epochenanzahl, sofern ein Netz als nicht vielversprechend interpretiert wird.
Internetquelle
Name, Vorname (Erscheinungsjahr): Vollständiger Titel, [online] direkter Link [Datum des
Abrufs].  

Q1: Jasper Snoek, Hugo Larochelle, Ryan P. Adams (2012): Practical Bayesian Optimization of Machine Learning Algorithms [online] https://papers.nips.cc/paper_files/paper/2012/hash/05311655a15b75fab86956663e1819cd-Abstract.html[23.03.2026]  

Q2: Jochen Görtler, Rebecca Kehlbeck, Oliver Deussen (2019): A Visual Exploration of Gaussian Processes [online] https://distill.pub/2019/visual-exploration-gaussian-processes/

Zunächst erfolgt eine Aufteilung der Daten. Der Einfach- und Vergleichbarkeit wegen wird der gleiche Code wie in 05_Neural_Network_Felix.ipynb verwendet.

In [ ]:
###Code übernommen aus 05_Neural_Network_Felix.ipynb
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.linear_model import LinearRegression

from utils.data import load_and_clean_data, get_train_test_split
from utils.evaluation import evaluate_predictions, add_result, evaluate_test_predictions
from utils.plotting import plot_features_vs_target, plot_predicted_vs_actual, plot_residuals, save_fig

plt.rcParams['figure.dpi'] = 100
%matplotlib inline

print(f"TensorFlow Version: {tf.__version__}")
print(f"Numpy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Matplotlib Version: {plt.matplotlib.__version__}")
print(f"python Version: {sys.version}")



In [ ]:
###Code übernommen aus 05_Neural_Network_Felix.ipynb
def build_model(hidden_layers, activation='relu', learning_rate=0.001):
    """Erstelle ein Sequential-Modell mit gegebener Architektur."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
    
    model.add(layers.Dense(1))  # Regression Output
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    return model


def train_and_evaluate(model, name, x_train, y_train, x_val, y_val, x_test, y_test,
                       epochs=100, batch_size=32, verbose=0):

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
    )

    y_train_pred = model.predict(x_train, verbose=0).flatten()
    y_test_pred = model.predict(x_test, verbose=0).flatten()

    result = evaluate_predictions(y_train, y_train_pred, y_test, y_test_pred, name)
    add_result(result)

    return history, result

### Daten laden und transformieren + skalieren
Die Daten werden in Trainings und Testdaten gesplitet. Ein Teil Trainingsdatensatzes wird als Validierungsdatensatz verwendet. Der aufgeteilte Datensatz erhält dann folgende Umfänge und Features:

In [ ]:
###Code übernommen aus 05_Neural_Network_Felix.ipynb
df = load_and_clean_data()
X_train, X_test, y_train, y_test, feature_names = get_train_test_split(df)

# Validierungssplit aus Trainingsdaten
val_split = int(0.8 * len(X_train))
X_val, y_val = X_train[val_split:], y_train[val_split:]
X_train, y_train = X_train[:val_split], y_train[:val_split]

print(f"Training:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"Features:   {feature_names}")
###Code übernommen aus 05_Neural_Network_Felix.ipynb
X_train_standard, X_test_standard, y_train_standard, y_test_standard,scaler, feature_names = X_train, X_test, y_train, y_test, scaler, feature_names = get_train_test_split(df, scaler='standard')

val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard, y_train_standard = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standard.shape}")
print(f"Features:   {feature_names}")
###Code übernommen aus 05_Neural_Network_Felix.ipynb
X_train_min0_max1, X_test_min0_max1, y_train_min0_max1, y_test_min0_max1, scaler, feature_names = get_train_test_split(df, scaler='minmax')

val_split = int(0.8 * len(X_train_min0_max1))
X_val_min0_max1, y_val_min0_max1 = X_train_min0_max1[val_split:], y_train_min0_max1[val_split:]
X_train_min0_max1, y_train_min0_max1 = X_train_min0_max1[:val_split], y_train_min0_max1[:val_split]

print(f"Training:   {X_train_min0_max1.shape}")
print(f"Validation: {X_val_min0_max1.shape}")
print(f"Test:       {X_test_min0_max1.shape}")
print(f"Features:   {feature_names}")



## Neuronale Netze trainieren


Die bisherige Analyse ergab eine optimale Lernrate zwischen .001 und 0.01. Der nächste Schritt zur effizienteren Suche eines geeigneten NN ist statt random-, grid search die Verwendung der in keras_tuner enthaltenen Funktion zu Bayesianischer Optimierung. 
Hierzu half der KI-Promt: "Welche Möglichkeiten zur Simultan-optimierung sind für ein Neuronales Netz nach Versuchen mit grid-, und random search noch geeignet?".
Der nächste logische Schritt bedurfte einiges an Recherche. 

In [ ]:

import keras_tuner as kt
from sklearn.metrics import r2_score

# Reminder: falls nicht funktional, skalieren auf -1 bis 1 probieren


### Information zu Keras-tuner
Quelle: Luca Invernizzi, James Long, Francois Chollet, Tom O'Malley, Haifeng Jin (2019): Getting started with KerasTuner https://keras.io/keras_tuner/getting_started/

Idee Nutzung "EarlyStopping" ist eine Funktion zur drastischen Reduktion der Rechenzeit durch abbrechen nicht zielführender Stränge.
Es wurden Standartwerte oder von KI empfohlene Werte für Berechnung verwendet.

In [ ]:
#learning rate (inzwischen funktional)
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=0, min_lr=1e-6
)
early_stopping = keras.callbacks.EarlyStopping(                             
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1       #mit behalten der besten gewichte für Reproduktion
)

# Bayesianischer Tuner
tuner = kt.BayesianOptimization(
    build_tuner_model,
    objective='val_loss',     # MSE auf den Validierungsdaten minimieren
    max_trials=15,            # Anzahl der zu testenden Kombinationen
    num_initial_points=5,     # zunächst 5 Random-Suchen um Bereich "kennen zu lernen"
    directory='tuning_dir',     
    project_name='california_housing_bayes',
    overwrite=True            # auf false, wenn man abgebrochenes tuning fortsetzen will
)

print("Starte Hyperparameter-Tuning...")
tuner.search(
    X_train_standard, y_train_standard,
    validation_data=(X_val_standard, y_val_standard),
    epochs=150,               # Maximal Epochen, meist greift EarlyStopping früher
    batch_size=32,            # Fixiert auf 32, da dies im Notebook stabil lief
    callbacks=[early_stopping, lr_scheduler],
    verbose=1                 #anzeige eines Ladebalkens
)

# Beste Parameter werden hier ausgegeben
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nBeste Hyperparameter gefunden:")
print(f"- Aktivierung: {best_hps.get('activation')}")
print(f"- Layer 1 Neuronen: {best_hps.get('units_1')}")
print(f"- Layer 1 Dropout: {best_hps.get('dropout_1')}")
print(f"- Layer 2 Neuronen: {best_hps.get('units_2')}")
print(f"- Lernrate: {best_hps.get('learning_rate'):.5f}")

### Stärkstes Einzelmodell

In [ ]:
# Bestes Modell mit den optimalen Parametern bauen
best_model = tuner.hypermodel.build(best_hps)

print("\nTrainiere finales Modell mit den besten Parametern...")
history = best_model.fit(
    X_train_standard, y_train_standard,
    validation_data=(X_val_standard, y_val_standard),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping, lr_scheduler],
    verbose=0 # Auf 1 setzen für Epochen-Output
)

# Vorhersagen auf den Testdaten machen
y_test_pred = best_model.predict(X_test_standard).flatten()

mse_test, mae_test = best_model.evaluate(X_test_standard, y_test_standard, verbose=0) #verbose auf 1 für aktuellen stand
r2_test = r2_score(y_test_standard, y_test_pred)

print(f"\n--- Finale Test-Ergebnisse ---")
print(f"L2 Loss (MSE): {mse_test:.4f}")
print(f"MAE:           {mae_test:.4f}")
print(f"R² Score:      {r2_test:.4f}")

#plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Trainings- und Validierungsverlauf (L2 Loss / MSE)
axes[0].plot(history.history['loss'], label='Train Loss (MSE)')
axes[0].plot(history.history['val_loss'], label='Val Loss (MSE)')
axes[0].set_title('Modell Loss über Epochen')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_xlabel('Epoche')
axes[0].legend()
axes[0].grid(True)

# Plot 2: Scatterplot Vorhersage vs. Tatsächliche Werte
axes[1].scatter(y_test_standard, y_test_pred, alpha=0.4, color='blue')
# Referenzlinie (Perfekte Vorhersage)
min_val = min(np.min(y_test_standard), np.min(y_test_pred))
max_val = max(np.max(y_test_standard), np.max(y_test_pred))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfekte Vorhersage')

axes[1].set_title(f'Testdaten: Tatsächlich vs. Vorhergesagt (R² = {r2_test:.3f})')
axes[1].set_xlabel('Tatsächliche Preise (standardisiert)')
axes[1].set_ylabel('Vorhergesagte Preise (standardisiert)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Modell speichern
model_path = "best_keras_tuner_model.h5"
best_model.save(model_path)
print(f"\nBestes Modell gespeichert unter: {model_path}")

In [ ]:
import os
from datetime import datetime


base_model_dir = "gespeicherte_modelle"
os.makedirs(base_model_dir, exist_ok=True)

zeitstempel = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_run_dir = os.path.join(base_model_dir, f"run_{zeitstempel}")
os.makedirs(current_run_dir, exist_ok=True)

print(f"Top-Modelle dieses Durchlaufs landen in: {current_run_dir}\n")

# vorher trainierte Modelle aufrufen
num_ensemble_models = 5
top_models = tuner.get_best_models(num_models=num_ensemble_models)

all_preds = []

for i, model in enumerate(top_models):
    # Name und Pfad für die Datei
    m_name = f"modell_platz_{i+1}.h5"
    full_path = os.path.join(current_run_dir, m_name)
    
    # Speichern
    model.save(full_path)
    
    # Vorhersage für das Testset berechnen und sammeln
    preds = model.predict(X_test_standard, verbose=0).flatten()
    all_preds.append(preds)
    print(f" -> {m_name} erfolgreich gespeichert.")

#ENSEMBLE BILDEN 
# Durchschnitt aller Vorhersagen bilden
y_ensemble_pred = np.mean(all_preds, axis=0)

# Metriken berechnen
ensemble_r2 = r2_score(y_test_standard, y_ensemble_pred)
ensemble_mae = np.mean(np.abs(y_test_standard - y_ensemble_pred))

print(f"\n--- Finale Ensemble Test-Ergebnisse ---")
print(f"Ensemble MAE:      {ensemble_mae:.4f}")
print(f"Ensemble R² Score: {ensemble_r2:.4f}")

#RESIDUEN PLOTTEN
plt.figure(figsize=(10, 6))
residuen_ensemble = y_test_standard - y_ensemble_pred

plt.hist(residuen_ensemble, bins=50, alpha=0.7, color='purple', label=f'Ensemble ({num_ensemble_models} Modelle)')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.title(f'Residuen-Histogramm des Ensembles (R² = {ensemble_r2:.4f})')
plt.xlabel('Residuen (Wahrer Preis - Vorhersage)')
plt.ylabel('Häufigkeit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Verändertes Featureengeneerning

Ki Prmopt: "Wie kann ich die Performance des R^2 durch Featureengeneerning auf über 0.8 bringen?"
Idee:  Features vor dem Training des neuronalen Netzes miteinander verrrechnen
 - bereits geschehen bei AveRooms und AveOccup
 - Nun auch mit "AveBedrms"/"AveRooms" = "Bedrooms_per_Room"
   - Tipp von Ki: Population logarithmisch betrachten und Nullfehler vermeiden durch +1
 - restlicher code zunächst unverändert
   - Ergebnis: 0.7922
 - Anpassung der Architektur: geringere Tiefe zur unterbindung des      gradient vanishing und zur Stabilisierunh
 - verringerte Rechenzeit nun nutzen für noch höhere Breite
   - durch Dropout auswendig lernen verhindern
- entfernen der Kommentare zu bereits erklärten Funktionen

#


In [ ]:

df = load_and_clean_data()

# geografische Features
# Koordinaten der Zentren (Breitengrad, Längengrad)
sf_lat, sf_lon = 37.7749, -122.4194
la_lat, la_lon = 34.0522, -118.2437
# Distanz zu San francisco und Los Angeles
df['Dist_to_SF'] = np.sqrt((df['Latitude'] - sf_lat)**2 + (df['Longitude'] - sf_lon)**2)
df['Dist_to_LA'] = np.sqrt((df['Latitude'] - la_lat)**2 + (df['Longitude'] - la_lon)**2)
#Kombinationsfeature
df['Lat_x_Lon'] = df['Latitude'] * df['Longitude']

df['Bedrooms_per_Room'] = df['AveBedrms'] / df['AveRooms']

# Logarithmus der Bevölkerung 
df['Log_Population'] = np.log(df['Population'] + 1) # +1 um Nullfehler auszuschließen
# ------------------------------------------
X_train, X_test, y_train, y_test, feature_names = get_train_test_split(df)

# Validierungssplit aus Trainingsdaten
val_split = int(0.8 * len(X_train))
X_val, y_val = X_train[val_split:], y_train[val_split:]
X_train, y_train = X_train[:val_split], y_train[:val_split]

print(f"Training:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"Features:   {feature_names}")

X_train_standard, X_test_standard, y_train_standard, y_test_standard,scaler, feature_names = X_train, X_test, y_train, y_test, scaler, feature_names = get_train_test_split(df, scaler='standard')

val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard, y_train_standard = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standard.shape}")
print(f"Features:   {feature_names}")

X_train_min0_max1, X_test_min0_max1, y_train_min0_max1, y_test_min0_max1, scaler, feature_names = get_train_test_split(df, scaler='minmax')

val_split = int(0.8 * len(X_train_min0_max1))
X_val_min0_max1, y_val_min0_max1 = X_train_min0_max1[val_split:], y_train_min0_max1[val_split:]
X_train_min0_max1, y_train_min0_max1 = X_train_min0_max1[:val_split], y_train_min0_max1[:val_split]

print(f"Training:   {X_train_min0_max1.shape}")
print(f"Validation: {X_val_min0_max1.shape}")
print(f"Test:       {X_test_min0_max1.shape}")
print(f"Features:   {feature_names}")



In [ ]:
# redundant, falls zuvor nicht ausgeführt 
import keras_tuner as kt
from sklearn.metrics import r2_score


In [ ]:

lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=0, min_lr=1e-6
)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

# Bayesianischer Tuner
tuner = kt.BayesianOptimization(
    build_tuner_model,
    objective='val_loss',     #MSE auf den Validierungsdaten minimieren
    max_trials=30,            # Anzahl der zu testenden Kombinationen
    num_initial_points=5,     
    directory='tuning_dir',
    project_name='california_housing_bayes',
    overwrite=True            
)

print("Starte Hyperparameter-Tuning...")
tuner.search(
    X_train_standard, y_train_standard,
    validation_data=(X_val_standard, y_val_standard),
    epochs=150,               # Maximal Epochen, meist greift EarlyStopping früher
    batch_size=32,            
    callbacks=[early_stopping, lr_scheduler],
    verbose=1                 #anzeige eines Ladebalkens
)

# Beste Parameter ausgeben
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nBeste Hyperparameter gefunden:")
print(f"- Aktivierung: {best_hps.get('activation')}")
print(f"- Layer 1 Neuronen: {best_hps.get('units_1')}")
print(f"- Layer 1 Dropout: {best_hps.get('dropout_1')}")
print(f"- Layer 2 Neuronen: {best_hps.get('units_2')}")
print(f"- Lernrate: {best_hps.get('learning_rate'):.5f}")

### Stärkstes Einzelmodell

In [ ]:
# Bestes Modell mit den optimalen Parametern bauen
best_model = tuner.hypermodel.build(best_hps)

print("\nTrainiere finales Modell mit den besten Parametern...")
history = best_model.fit(
    X_train_standard, y_train_standard,
    validation_data=(X_val_standard, y_val_standard),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping, lr_scheduler],
    verbose=0 # Auf 1 setzen für Epochen-Output
)

# Vorhersagen auf den Testdaten machen
y_test_pred = best_model.predict(X_test_standard).flatten()

mse_test, mae_test = best_model.evaluate(X_test_standard, y_test_standard, verbose=0)
r2_test = r2_score(y_test_standard, y_test_pred)

print(f"\n--- Finale Test-Ergebnisse ---")
print(f"L2 Loss (MSE): {mse_test:.4f}")
print(f"MAE:           {mae_test:.4f}")
print(f"R² Score:      {r2_test:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Trainings- und Validierungsverlauf (L2 Loss / MSE)
axes[0].plot(history.history['loss'], label='Train Loss (MSE)')
axes[0].plot(history.history['val_loss'], label='Val Loss (MSE)')
axes[0].set_title('Modell Loss über Epochen')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_xlabel('Epoche')
axes[0].legend()
axes[0].grid(True)

# Plot Scatterplot Vorhersage vs. Tatsächliche Werte
axes[1].scatter(y_test_standard, y_test_pred, alpha=0.4, color='blue')
# Referenzlinie (Perfekte Vorhersage)
min_val = min(np.min(y_test_standard), np.min(y_test_pred))
max_val = max(np.max(y_test_standard), np.max(y_test_pred))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfekte Vorhersage')

axes[1].set_title(f'Testdaten: Tatsächlich vs. Vorhergesagt (R² = {r2_test:.3f})')
axes[1].set_xlabel('Tatsächliche Preise (standardisiert)')
axes[1].set_ylabel('Vorhergesagte Preise (standardisiert)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Modell abspeichern
model_path = "best_keras_tuner_model.h5"
best_model.save(model_path)
print(f"\nBestes Modell gespeichert unter: {model_path}")

Im linken Plot ist die Abnahme des Loss über die Epochen zu sehen. Es ist zu erkennen, dass weitere Epochen realtiv schnell keiner weiteren Verbesserung der Ergebnisse auf die Validierungsdaten helfen. 
-> mehr Epochen helfen einem besseren Modell nicht weiter
-> kein Problem durch baysiansische Optimierung werden die meisten Durchgänge lang vorher abgebrochen


In [ ]:
base_model_dir = r"C:\Users\fneum\Documents\Uni\Physik\3.Semester\Kuenstliche Intelligenz\Projekt\ki-308\results\gespeicherte_modelle"
os.makedirs(base_model_dir, exist_ok=True)

zeitstempel = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_run_dir = os.path.join(base_model_dir, f"run_{zeitstempel}")
os.makedirs(current_run_dir, exist_ok=True)

print(f"Top-Modelle dieses Durchlaufs landen in: {current_run_dir}\n")

num_ensemble_models = 5
top_models = tuner.get_best_models(num_models=num_ensemble_models)

all_preds = []

for i, model in enumerate(top_models):
    # Name und Pfad für die Datei
    m_name = f"modell_platz_{i+1}.h5"
    full_path = os.path.join(current_run_dir, m_name)
    
    # Speichern
    model.save(full_path)
    
    # Vorhersage für das Testset berechnen und sammeln
    preds = model.predict(X_test_standard, verbose=0).flatten()
    all_preds.append(preds)
    print(f" -> {m_name} erfolgreich gespeichert.")

# Durchschnitt aller Vorhersagen bilden
y_ensemble_pred = np.mean(all_preds, axis=0)

# Metriken berechnen
ensemble_r2 = r2_score(y_test_standard, y_ensemble_pred)
ensemble_mae = np.mean(np.abs(y_test_standard - y_ensemble_pred))

print(f"\n--- Finale Ensemble Test-Ergebnisse ---")
print(f"Ensemble MAE:      {ensemble_mae:.4f}")
print(f"Ensemble R² Score: {ensemble_r2:.4f}")


plt.figure(figsize=(10, 6))
residuen_ensemble = y_test_standard - y_ensemble_pred

plt.hist(residuen_ensemble, bins=50, alpha=0.7, color='purple', label=f'Ensemble ({num_ensemble_models} Modelle)')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.title(f'Residuen-Histogramm des Ensembles (R² = {ensemble_r2:.4f})')
plt.xlabel('Residuen (Wahrer Preis - Vorhersage)')
plt.ylabel('Häufigkeit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Erneutes laden und ausgeben des R^2-Wertes erfolgt in folgender zelle: 

In [ ]:
#Laden der Modelle:
#erste zeile mit jeweiligem zeitstempel
modell_ordner = r"C:\Users\fneum\Documents\Uni\Physik\3.Semester\Kuenstliche Intelligenz\Projekt\ki-308\results\gespeicherte_modelle\run_2026-03-20_21-29-14 R^2 = 0.7951"
dateien = ["modell_platz_1.h5", "modell_platz_2.h5", "modell_platz_3.h5", "modell_platz_4.h5", "modell_platz_5.h5"]

all_preds = []

# Jedes Modell einzeln laden und vorhersagen lassen
for datei in dateien:
    pfad = os.path.join(modell_ordner, datei)
    model = tf.keras.models.load_model(pfad, compile=False)
    
    # Vorhersage machen 
    preds = model.predict(X_test_standard).flatten()
    all_preds.append(preds)
    #print(f"{datei} geladen und berechnet.")

# Durchschnitt
y_ensemble_pred = np.mean(all_preds, axis=0)

ensemble_r2 = r2_score(y_test_standard, y_ensemble_pred)
#print(f"\nEnsemble R² Score: {ensemble_r2:.4f}")

plt.figure(figsize=(10, 10))
# Streudiagramm der Vorhersagen vs Wahre Werte
plt.scatter(y_test_standard, y_ensemble_pred, alpha=0.3, color='dodgerblue', label='Vorhersagen')

# Perfekte Vorhersage-Linie (Diagonale)
min_val = min(np.min(y_test_standard), np.min(y_ensemble_pred))
max_val = max(np.max(y_test_standard), np.max(y_ensemble_pred))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Perfekte Vorhersage')

plt.title(f'Wahre Preise vs. Ensemble-Vorhersage (R² = {ensemble_r2:.4f})', fontsize=14)
plt.xlabel('Wahre Preise (Standardisiert)', fontsize=12)
plt.ylabel('Vorhergesagte Preise (Standardisiert)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Stand 21.03.2026
- Die bisher stärkste Regression
- Idee:  Obwohl bereits als decisiontree und mit random forst von Björn geschehen, auch einen Random forest einführen. 
- Untersuchung der neu eingeführten Features auf ihren Nutzen für das NN anstatt "blind" Berechnungen zu starten

In [ ]:

df = load_and_clean_data()

# geografische Features
# Koordinaten der Zentren (Breitengrad, Längengrad)
sf_lat, sf_lon = 37.7749, -122.4194
la_lat, la_lon = 34.0522, -118.2437
# Distanz zu San francisco und Los Angeles
df['Dist_to_SF'] = np.sqrt((df['Latitude'] - sf_lat)**2 + (df['Longitude'] - sf_lon)**2)
df['Dist_to_LA'] = np.sqrt((df['Latitude'] - la_lat)**2 + (df['Longitude'] - la_lon)**2)
#Kombinationsfeature
df['Lat_x_Lon'] = df['Latitude'] * df['Longitude']

df['Bedrooms_per_Room'] = df['AveBedrms'] / df['AveRooms']

# Logarithmus der Bevölkerung 
df['Log_Population'] = np.log(df['Population'] + 1) # +1 um Nullfehler auszuschließen
# ------------------------------------------
X_train, X_test, y_train, y_test, feature_names = get_train_test_split(df)

# Validierungssplit aus Trainingsdaten
val_split = int(0.8 * len(X_train))
X_val, y_val = X_train[val_split:], y_train[val_split:]
X_train, y_train = X_train[:val_split], y_train[:val_split]

print(f"Training:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"Features:   {feature_names}")

X_train_standard, X_test_standard, y_train_standard, y_test_standard,scaler, feature_names = X_train, X_test, y_train, y_test, scaler, feature_names = get_train_test_split(df, scaler='standard')

val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard, y_train_standard = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standard.shape}")
print(f"Features:   {feature_names}")

X_train_min0_max1, X_test_min0_max1, y_train_min0_max1, y_test_min0_max1, scaler, feature_names = get_train_test_split(df, scaler='minmax')

val_split = int(0.8 * len(X_train_min0_max1))
X_val_min0_max1, y_val_min0_max1 = X_train_min0_max1[val_split:], y_train_min0_max1[val_split:]
X_train_min0_max1, y_train_min0_max1 = X_train_min0_max1[:val_split], y_train_min0_max1[:val_split]

print(f"Training:   {X_train_min0_max1.shape}")
print(f"Validation: {X_val_min0_max1.shape}")
print(f"Test:       {X_test_min0_max1.shape}")
print(f"Features:   {feature_names}")



## Random Forest

In [ ]:

from sklearn.ensemble import RandomForestRegressor

if isinstance(y_train, pd.DataFrame) or isinstance(y_train, pd.Series):
    y_train_rf = y_train.values.ravel()
else:
    y_train_rf = y_train.ravel()


# n_jobs=-1: alle Kerne desProzessors nutzen
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

rf.fit(X_train, y_train_rf)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1] # Absteigend sortieren

#absteigende Reihenfolge
sorted_features = [feature_names[i] for i in indices]
#Plot
plt.figure(figsize=(12, 6))
plt.title("Feature Importance: Welchen Einfluss haben die einzelnen Merkmale?", fontsize=14, fontweight='bold')

# Balkendiagramm 
plt.bar(range(X_train.shape[1]), importances[indices], color='lightblue', edgecolor='black')

# Achsenbeschriftungen anpassen
plt.xticks(range(X_train.shape[1]), sorted_features, rotation=45, ha='right', fontsize=11)
plt.ylabel("Relative Wichtigkeit (Gini Importance)", fontsize=12)

# exakte Werte hinzufügen
for i, val in enumerate(importances[indices]):
    plt.text(i, val + 0.005, f"{val:.3f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

y_rf_pred = rf.predict(X_test) 
print(f"Random Forest R² Score auf Testdaten: {r2_score(y_test, y_rf_pred):.4f}")

Zu sehen sind die Feature Importances und der R^2.
Der Random Forest erzielt hierbei ein minimal besseres 


### Training mit optimierten Features

- nun findet das Training des NN, mit bekanntem Code, unter Ausschluss der schwächsten Features zur Beschreibung des Datensatzes statt.


In [ ]:
# Ausgeschlossene Features: Population, Log_Population
df.drop(columns=["Population", "Log_Population"])

#ab hier zusammengekürzter bekannter Code
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=0, min_lr=1e-6
)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

# Bayesianischer Tuner
tuner = kt.BayesianOptimization(
    build_tuner_model,
    objective='val_loss',     
    max_trials=30,           
    num_initial_points=5,     
    directory='tuning_dir',
    project_name='california_housing_bayes',
    overwrite=True            
)
tuner.search(
    X_train_standard, y_train_standard,
    validation_data=(X_val_standard, y_val_standard),
    epochs=150,             
    batch_size=32,            
    callbacks=[early_stopping, lr_scheduler],
    verbose=1               
)

# Beste Parameter ausgeben
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nBeste Hyperparameter gefunden:")
print(f"- Aktivierung: {best_hps.get('activation')}")
print(f"- Layer 1 Neuronen: {best_hps.get('units_1')}")
print(f"- Layer 1 Dropout: {best_hps.get('dropout_1')}")
print(f"- Layer 2 Neuronen: {best_hps.get('units_2')}")
print(f"- Lernrate: {best_hps.get('learning_rate'):.5f}")
# Bestes Modell mit den optimalen Parametern bauen
best_model = tuner.hypermodel.build(best_hps)

print("\nTrainiere finales Modell mit den besten Parametern...")
history = best_model.fit(
    X_train_standard, y_train_standard,
    validation_data=(X_val_standard, y_val_standard),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping, lr_scheduler],
    verbose=0 
)

# Vorhersagen auf den Testdaten machen
y_test_pred = best_model.predict(X_test_standard).flatten()

mse_test, mae_test = best_model.evaluate(X_test_standard, y_test_standard, verbose=0)
r2_test = r2_score(y_test_standard, y_test_pred)

print(f"\n--- Finale Test-Ergebnisse ---")
print(f"L2 Loss (MSE): {mse_test:.4f}")
print(f"MAE:           {mae_test:.4f}")
print(f"R² Score:      {r2_test:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Trainings- und Validierungsverlauf (L2 Loss / MSE)
axes[0].plot(history.history['loss'], label='Train Loss (MSE)')
axes[0].plot(history.history['val_loss'], label='Val Loss (MSE)')
axes[0].set_title('Modell Loss über Epochen')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_xlabel('Epoche')
axes[0].legend()
axes[0].grid(True)

# Plot Scatterplot Vorhersage vs. Tatsächliche Werte
axes[1].scatter(y_test_standard, y_test_pred, alpha=0.4, color='blue')
# Referenzlinie (Perfekte Vorhersage)
min_val = min(np.min(y_test_standard), np.min(y_test_pred))
max_val = max(np.max(y_test_standard), np.max(y_test_pred))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfekte Vorhersage')

axes[1].set_title(f'Testdaten: Tatsächlich vs. Vorhergesagt (R² = {r2_test:.3f})')
axes[1].set_xlabel('Tatsächliche Preise (standardisiert)')
axes[1].set_ylabel('Vorhergesagte Preise (standardisiert)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Modell abspeichern
model_path = "best_keras_tuner_model.h5"
best_model.save(model_path)

base_model_dir = r"C:\Users\fneum\Documents\Uni\Physik\3.Semester\Kuenstliche Intelligenz\Projekt\ki-308\results\gespeicherte_modelle"
os.makedirs(base_model_dir, exist_ok=True)

zeitstempel = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_run_dir = os.path.join(base_model_dir, f"run_{zeitstempel}")
os.makedirs(current_run_dir, exist_ok=True)

print(f"Top-Modelle dieses Durchlaufs landen in: {current_run_dir}\n")

num_ensemble_models = 5
top_models = tuner.get_best_models(num_models=num_ensemble_models)

all_preds = []

for i, model in enumerate(top_models):
    # Name und Pfad für die Datei
    m_name = f"modell_platz_{i+1}.h5"
    full_path = os.path.join(current_run_dir, m_name)
    
    # Speichern
    model.save(full_path)
    
    # Vorhersage für das Testset berechnen und sammeln
    preds = model.predict(X_test_standard, verbose=0).flatten()
    all_preds.append(preds)
    print(f" -> {m_name} erfolgreich gespeichert.")

# Durchschnitt aller Vorhersagen bilden
y_ensemble_pred = np.mean(all_preds, axis=0)

# Metriken berechnen
ensemble_r2 = r2_score(y_test_standard, y_ensemble_pred)
ensemble_mae = np.mean(np.abs(y_test_standard - y_ensemble_pred))

print(f"\n--- Finale Ensemble Test-Ergebnisse ---")
print(f"Ensemble MAE:      {ensemble_mae:.4f}")
print(f"Ensemble R² Score: {ensemble_r2:.4f}")


plt.figure(figsize=(10, 6))
residuen_ensemble = y_test_standard - y_ensemble_pred

plt.hist(residuen_ensemble, bins=50, alpha=0.7, color='purple', label=f'Ensemble ({num_ensemble_models} Modelle)')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.title(f'Residuen-Histogramm des Ensembles (R² = {ensemble_r2:.4f})')
plt.xlabel('Residuen (Wahrer Preis - Vorhersage)')
plt.ylabel('Häufigkeit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Ergebnisse Feature drop 
- zunächst ohne "Population" und "Log_Population" 
  - keine Verbesserung bei R^2 und Loss...
Entscheidung:
- da die beiden schwächsten Features noch immer über 1% liefern, beibehalten für das NN. Neuronale Netze können auch aus kleinen Signalen noch Muster schließen, daher hilft es nicht sie zu beschneiden. Dies würde vermutlich erst weit unter einem Prozent zu einem spürbaren Ergebnis führen, da hier das Rauschen dem Informationsgewinn überwiegt.

Dieser Eindruck und auch die sehr geringe Hilfe der neu hinzugefügten Features wird zusätzlich durch die Korrealtionsanalyse unterstützt: 

In [ ]:
from utils.plotting import plot_correlation_heatmap
fig, ax = plot_correlation_heatmap(df, save_name="eda_correlation_heatmap")
plt.show()

Laden des besten Modells 

In [ ]:
#Laden der Modelle:
import os
#erste zeile mit jeweiligem zeitstempel
modell_ordner = r"C:\Users\fneum\Documents\Uni\Physik\3.Semester\Kuenstliche Intelligenz\Projekt\ki-308\results\gespeicherte_modelle\run_2026-03-20_21-29-14 R^2 = 0.7951"
dateien = ["modell_platz_1.h5", "modell_platz_2.h5", "modell_platz_3.h5", "modell_platz_4.h5", "modell_platz_5.h5"]

all_preds = []

# Jedes Modell einzeln laden und vorhersagen lassen
for datei in dateien:
    pfad = os.path.join(modell_ordner, datei)
    model = tf.keras.models.load_model(pfad, compile=False)
    
    # Vorhersage machen 
    preds = model.predict(X_test_standard).flatten()
    all_preds.append(preds)
    #print(f"{datei} geladen und berechnet.")

# Durchschnitt
y_ensemble_pred = np.mean(all_preds, axis=0)

ensemble_r2 = r2_score(y_test_standard, y_ensemble_pred)
#print(f"\nEnsemble R² Score: {ensemble_r2:.4f}")

plt.figure(figsize=(10, 10))
# Streudiagramm der Vorhersagen vs Wahre Werte
plt.scatter(y_test_standard, y_ensemble_pred, alpha=0.3, color='dodgerblue', label='Vorhersagen')

# Perfekte Vorhersage-Linie (Diagonale)
min_val = min(np.min(y_test_standard), np.min(y_ensemble_pred))
max_val = max(np.max(y_test_standard), np.max(y_ensemble_pred))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Perfekte Vorhersage')

plt.title(f'Wahre Preise vs. Ensemble-Vorhersage (R² = {ensemble_r2:.4f})', fontsize=14)
plt.xlabel('Wahre Preise (Standardisiert)', fontsize=12)
plt.ylabel('Vorhergesagte Preise (Standardisiert)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Vorhersagen (y_ensemble_pred) und wahre Werte (y_test) vergleichen
r2 = r2_score(y_test_standard, y_ensemble_pred)
mae = mean_absolute_error(y_test_standard, y_ensemble_pred)
rmse = np.sqrt(mean_squared_error(y_test_standard, y_ensemble_pred))

print("--- Modell Evaluierung ---")
print(f"R² Score: {r2:.4f}")
print(f"MAE:      {mae:.4f}")
print(f"RMSE:     {rmse:.4f}")


--- Modell Evaluierung ---  
R² Score: 0.7951           
MAE:      0.2921            
RMSE:     0.4315             

## Zusammenfassung
- Hinzufügen des Features ("AveBdrm") ergab keine Verbesserung
  - allerdings zuvor schon sehr guter Wert und ähnliche Werte konnten erreicht werden
- hinzufügen leaky relu ebenfalls keine deutliche Verbesserung

Stand 22.03.2026